# A1. Data inventory (Table 1 and Fig 1c input)

Table 1 / Fig 1b input: every base variable of the candidate pool, both cities.
Period and completeness are COMPUTED from the master datasets; source, resolution and
units are curated from the pipeline code (notes column flags anything uncertain).
Completeness = valid days / days in that city's MODELLING window (review item R8: the earlier
version used the full data-download range, 2014-2025 for Bogotá, which understated CO and NO's
completeness -- they run 2014-2022 with no gaps, but read as 75% complete against a denominator
that includes 2023-2025, years the equations never see. The window here is the one actually used
by feature selection / PySR for each city.)

In [1]:
# Settings (edit here)
MODELLING_WINDOW = {"Bogota": ("2014-01-01", "2022-12-31"), "Medellin": ("2013-01-01", "2022-12-31")}

In [2]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()          # notebooks/ folder
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import sys
from pathlib import Path
import numpy as np, pandas as pd
from style import *   # PM25, O3, NAVY, GREY, RED, LIGHT, ROOT, DATA_OUT, save_fig, panel_label

In [3]:
# Analysis
ROOT = ROOT
D = ROOT / "Data/processed/colombia"
M = {"Bogota": [D/"bogota/bogota_master_dataset.parquet", D/"bogota/bogota_o3_master_dataset.parquet"],
     "Medellin": [D/"medellin/medellin_master_dataset.parquet", D/"medellin/medellin_o3_master_dataset.parquet"]}
frames = {}
for c, ps in M.items():
    a, b = [pd.read_parquet(p) for p in ps]
    a["date"] = pd.to_datetime(a["date"]); b["date"] = pd.to_datetime(b["date"])
    w0, w1 = MODELLING_WINDOW[c]
    a = a[(a.date >= w0) & (a.date <= w1)].reset_index(drop=True); b = b[(b.date >= w0) & (b.date <= w1)].reset_index(drop=True)
    frames[c] = (a, b)

# variable, label, source, resolution, units, note
INV = [
 ("pm25", "PM2.5", "SISAIRE regulatory network (network mean of QC'd stations); Medellin: Antioquia SISAIRE (Aburra Valley) with calibrated Hourmean fallback", "daily mean of station means", "ug m-3", ""),
 ("pm10", "PM10", "SISAIRE (as above)", "daily mean", "ug m-3", "excluded from PM2.5 predictors (PM2.5 is a subset of PM10)"),
 ("no2", "NO2", "SISAIRE / Hourmean network", "daily mean", "ug m-3", "unit as delivered by the network; not re-verified"),
 ("no", "NO", "hourly network files (Bogota: USA_Ozone DATA_ORIGINAL; Medellin: Hourmean)", "daily mean of hourly", "Bogota ppb (verified, A4); Medellin ug m-3 (assumed, not independently verified)", "E4 (2026-09-22): units are NOT the same in both cities"),
 ("no_sisaire", "NO (network)", "hourly network files (Medellin only)", "daily mean of hourly", "ug m-3 (assumed, not independently verified)", "2026-09-24: second, independent NO record from the Aburra Valley network, distinct from the 'no' row above; selected into Medellin's reported ozone equation"),
 ("co", "CO", "hourly network files", "daily mean of hourly", "ppm (probable, A4; not independently verified -- no SISAIRE CO series to cross-check)", "2026-09-24: now measured in the Aburra Valley network too, but its qualifying station-years (2009-2011) predate both modelling windows, so it is unusable for either target regardless of coverage"),
 ("so2", "SO2", "hourly network files", "daily mean of hourly", "ug m-3", "2026-09-24: now measured in the Aburra Valley network too; clears the 70% coverage filter for Medellin ozone (not PM2.5) but is not selected into either reported equation"),
 ("radiation_station", "solar radiation (station)", "hourly network files (Medellin only)", "daily mean of hourly", "W m-2", "2026-09-24: distinct from the ERA5 radiation_mean row below; selected into Medellin's reported ozone equation"),
 ("temp_10cm", "near-surface (10 cm) air temperature", "hourly network files (Medellin only)", "daily mean of hourly", "degC", "2026-09-24: in ozone's candidate pool but not selected; excluded from PM2.5's pool by coverage"),
 ("precip_station", "precipitation (station)", "hourly network files (Medellin only)", "daily mean of hourly", "mm day-1", "2026-09-24: in ozone's candidate pool but not selected; excluded from PM2.5's pool by coverage"),
 ("o3", "O3 (MDA8)", "hourly network files -> daily maximum 8-h running mean (Bogota: 20 stations; Medellin: Hourmean network mean)", "daily MDA8", "Bogota ppb (verified, A4); Medellin ug m-3 (verified, A4)", "E4 (2026-09-22): the two cities' O3 are NOT the same unit -- do not compare magnitudes directly. SISAIRE daily O3 is a plain daily mean, not MDA8, so not used"),
 ("pressure", "surface pressure", "Bogota: SISAIRE station; Medellin: ERA5 (station sensors miscalibrated, dropped)", "daily mean", "Bogota mmHg (563 mmHg = 751 hPa); Medellin hPa", "units differ between cities; qc.py comment says hPa, which is wrong for Bogota"),
 ("temp_2m", "air temperature", "Bogota: SISAIRE stations; Medellin: station network calibrated onto ERA5 where missing", "daily mean", "degC", ""),
 ("rh_2m", "relative humidity", "as above", "daily mean", "%", ""),
 ("wind_speed", "wind speed (station)", "as above", "daily mean", "m s-1", ""),
 ("wind_dir", "wind direction (station)", "as above", "daily vector mean", "degrees", ""),
 ("blh_mean", "boundary-layer height", "ERA5 single levels", "hourly -> daily mean", "m", "Medellin ERA5 features start 2014-12-31 (download 2015-2022)"),
 ("radiation_mean", "surface solar radiation", "ERA5 ssrd / 3600", "hourly -> daily mean", "W m-2", ""),
 ("precip_total_mm", "precipitation", "ERA5 total precipitation", "hourly -> daily sum", "mm day-1", ""),
 ("tcwv", "total column water vapour", "ERA5", "hourly -> daily mean", "kg m-2", ""),
 ("sensible_heat_flux", "surface sensible heat flux", "ERA5 sshf / 3600", "hourly -> daily mean", "W m-2", ""),
 ("tke_proxy", "turbulence proxy", "ERA5 10 m wind variance within the day, 0.5(var u + var v)", "hourly -> daily", "m2 s-2", "sub-daily variance, not measured TKE"),
 ("w700", "vertical velocity at 700 hPa", "ERA5 pressure levels", "hourly -> daily mean", "Pa s-1", "700 hPa is only ~45 hPa above Bogota's surface"),
 ("lts", "lower-tropospheric stability", "ERA5 (theta700 - theta_surface)", "hourly -> daily mean", "K", ""),
 ("hotspots_sector_E", "fire hotspots, east sector", "gridded hotspot product (0.75 deg), 45-degree sector within 1500 km of the city", "daily count", "count day-1", "Bogota record ends 2023-12-31; sectors are computed around each city"),
 ("hcho", "tropospheric HCHO column", "TROPOMI via Earth Engine, cloud fraction <= 0.3", "daily region mean", "mol m-2", "satellite era only; not used in any equation"),
]
rows = []
for var, label, src, res, unit, note in INV:
    row = dict(variable=var, description=label, source=src, native_resolution=res, units=unit, note=note)
    for city, (a, b) in frames.items():
        ndays = len(a)
        col = None
        for d in (a, b):
            if var in d.columns:
                col = d[["date", var]].dropna(); break
        if col is None:
            row.update({f"{city}_first": "", f"{city}_last": "", f"{city}_n_days": 0, f"{city}_completeness_pct": 0.0})
        else:
            row.update({f"{city}_first": col.date.min().date().isoformat(), f"{city}_last": col.date.max().date().isoformat(),
                        f"{city}_n_days": int(len(col)), f"{city}_completeness_pct": round(100 * len(col) / ndays, 1)})
        row[f"{city}_master_range"] = f"{a.date.min().date()} to {a.date.max().date()} ({ndays} d)"   # = the modelling window, filtered above
    rows.append(row)
t = pd.DataFrame(rows)
t.to_csv(DATA_OUT / "table1_data_inventory.csv", index=False)
print(t[["variable", "Bogota_first", "Bogota_completeness_pct", "Medellin_first", "Medellin_completeness_pct"]].to_string(index=False))
print(t.iloc[0][["Bogota_master_range", "Medellin_master_range"]].to_dict())

          variable Bogota_first  Bogota_completeness_pct Medellin_first  Medellin_completeness_pct
              pm25   2014-01-01                    100.0     2013-01-01                       96.1
              pm10   2014-01-01                    100.0     2013-01-01                       66.9
               no2   2014-01-01                     88.9     2013-01-01                       99.0
                no   2014-01-01                     99.9     2015-01-01                       78.6
        no_sisaire                                   0.0     2013-01-01                       68.9
                co   2014-01-01                     99.9            NaT                        0.0
               so2   2018-01-01                     52.4     2019-01-01                       35.2
 radiation_station                                   0.0     2013-01-01                       76.2
         temp_10cm                                   0.0     2019-01-01                       40.0
    precip